# Core Multi-League Dataset — EDA

> **Ligas:** LaLiga · Premier League · Bundesliga&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuente:** `core_validated.parquet` (de cada liga)

## Objetivos

- Evaluar la homogeneidad estructural entre competiciones (dimensiones, esquema y tipos de datos).
- Definir el **core dataset** común a las tres ligas.
- Analizar la completitud de datos y distribución de valores nulos por competición.
- Validar integridad estructural (consistencia FTR/HTR, duplicados, valores negativos).
- Determinar la viabilidad de un pipeline de limpieza unificado y estrategia de modelado multi-liga.

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---------|----------|
| 0 | Entorno y configuración | Librerías y rutas del proyecto |
| 1 | Lectura y organización | Carga de datasets core por liga |
| 2 | Comparación de esquemas | Dimensiones, core multi-league y tipos de dato |
| 3 | Evaluación de completitud | Nulos por liga y cobertura de odds |
| 4 | Validaciones de integridad | Coherencia, duplicados y valores inválidos |
| 5 | Conclusiones | Síntesis y consideraciones para limpieza |
| 6 | Exportación | Core multi-league validado |

##
---

## 0) Entorno y configuración

En esta sección se configuran las dependencias, librerías y parámetros globales necesarios para garantizar la reproducibilidad del análisis.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
import json
from IPython.display import display, Markdown

def _find_project_root(start: Path) -> Path:
    markers = {"config", "src", "data"}
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError(
        f"Raíz del proyecto no encontrada desde {start}. "
        f"Asegúrate de ejecutar el notebook desde dentro del proyecto."
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
with open(CONFIG_ROOT / "leagues.json") as f:
    ALL_LEAGUES = json.load(f)

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
CORE_MULTI_LEAGUE_VALIDATED_PATH = PROCESSED_ROOT / "core_multi_league_validated.parquet"
CORE_MULTI_LEAGUE_VALIDATED_SCHEMA_PATH = PROCESSED_ROOT / "core_multi_league_validated_schema.json"
PARQUET_NAME = "core_validated.parquet"
LEAGUES = [name for code, name in ALL_LEAGUES.items() if (PROCESSED_ROOT / name / PARQUET_NAME).exists()]

# Importación de funciones propias
from src.analysis import group_columns

---
##

## 1) Lectura y organización de datos

Carga de los datasets core procesados de cada liga (formato Parquet).

In [ ]:
dfs = {
    lg: pd.read_parquet(PROCESSED_ROOT / lg / PARQUET_NAME)
    for lg in LEAGUES
}

df_all = pd.concat(dfs.values(), ignore_index=True)

print(f"Datasets cargados: {len(dfs)}\n")
for lg in LEAGUES:
    rel_path = Path("data") / "processed" / lg / PARQUET_NAME
    print(f"  • {lg}: {rel_path}")

print(f"\nTotal de partidos: {sum(len(df) for df in dfs.values()):,}")


---
##

## 2) Comparación de esquemas

Se compara la estructura de los datasets de las tres ligas para identificar qué columnas son comunes, cuáles son específicas de cada liga y si los tipos de datos se mantienen consistentes.

### 2.1 Dimensiones de los datasets

Comparación del número de partidos y columnas por liga.

In [ ]:
dim_summary = pd.DataFrame([
    {
        "Liga": lg.capitalize(),
        "Partidos": len(df),
        "Columnas": len(df.columns)   
    }
    for lg, df in dfs.items()
])

display(dim_summary.style.hide(axis="index"))

### 2.2 Identificación del core multi-league

Variables comunes a las tres ligas que conforman el core consolidado.

In [ ]:
column_sets = {
    lg: set(df.columns)
    for lg, df in dfs.items()
}

core_consolidated = sorted(set.intersection(*column_sets.values()))
all_columns = sorted(set.union(*column_sets.values()))
league_specific = {
    lg: sorted(cols - set(core_consolidated))
    for lg, cols in column_sets.items()
}

print(f"Columnas totales (union de todas): {len(all_columns)}")
print(f"Columnas en core consolidado: {len(core_consolidated)}\n")

has_specific = any(len(cols) > 0 for cols in league_specific.values())

if has_specific:
    print("Columnas específicas por liga:\n")
    for lg, specific in league_specific.items():
        if specific:
            print(f"  • {lg.capitalize()}: {', '.join(specific)}")
else:
    print("Todas las columnas son comunes a las 3 ligas")

### 2.3 Consistencia de tipos de datos

Verificación de que las columnas del core multi-league mantienen el mismo tipo en las tres ligas.

In [ ]:
drift_cross_league = []

for col in core_consolidated:
    types_per_league = {lg: str(dfs[lg][col].dtype) for lg in LEAGUES}
    unique_types = set(types_per_league.values())
    
    if len(unique_types) > 1:
        drift_cross_league.append({
            "Column": col,
            **types_per_league
        })

drift_df = pd.DataFrame(drift_cross_league)

if len(drift_df) > 0:
    print(f"⚠ Columnas del core con drift de tipos entre ligas: {len(drift_df)}\n")
    display(drift_df.set_index("Column"))
else:
    print("Tipos de datos consistentes en el core consolidado")

### 2.4 Resumen del core multi-league

Clasificación y detalle de las variables comunes a las tres ligas.

#### 2.4.1 Resumen global

In [ ]:
core_consolidated_df = pd.DataFrame({"Variable": core_consolidated})

df_class = group_columns(core_consolidated_df["Variable"])

summary_groups = (
    df_class.groupby("Grupo", as_index=False)
    .agg(**{"Número de variables": ("Variable", "count")})
    .sort_values("Número de variables", ascending=False)
)

print(f"Core consolidado: {len(core_consolidated)} variables comunes a las 3 ligas\n")
display(summary_groups.style.hide(axis="index"))

#### 2.4.2 Detalle de variables por grupo

In [ ]:
detail_groups = (
    df_class.sort_values(["Grupo", "Variable"])
    .groupby("Grupo", as_index=False)
    .agg(Variables=("Variable", lambda x: "\n".join(x)))
)

display(
    detail_groups.style
    .set_properties(**{"white-space": "pre-wrap"})
    .hide(axis="index")
)

---
##

## 3) Evaluación de completitud

Evaluación de la calidad del core multi-league a través del análisis de valores nulos por liga, detección de casas de apuestas estables y síntesis de métricas globales de completitud.

### 3.1 Análisis de nulos por liga

Comparación del porcentaje de valores nulos en el core consolidado entre las tres ligas.

In [ ]:
null_summary = []

for lg, df in dfs.items():
    df_core = df[core_consolidated]
    total_cells = df_core.size
    total_nulls = df_core.isnull().sum().sum()
    null_pct = (total_nulls / total_cells) * 100
    
    null_summary.append({
        "Liga": lg.capitalize(),
        "Total nulos": total_nulls,
        "% nulos": round(null_pct, 2)
    })

null_df = pd.DataFrame(null_summary)
display(null_df.style.hide(axis="index"))

print(f"\nNulos totales en core consolidado: {null_df['Total nulos'].sum():,}")

### 3.2 Columnas con mayor proporción de nulos

Identificación de variables del core con completitud reducida por liga.

#### 3.2.1 Construcción de la comparativa

In [ ]:
null_data = {}

for lg, df in dfs.items():
    df_core = df[core_consolidated]
    col_nulls = df_core.isnull().sum()
    col_pct = (col_nulls / len(df_core)) * 100
    
    null_data[lg] = {
        col: col_pct[col]
        for col in core_consolidated
        if col_nulls[col] > 0
    }

all_null_cols = sorted(set().union(*[set(d.keys()) for d in null_data.values()]))

cols_in_all = [col for col in all_null_cols if all(col in null_data.get(lg, {}) for lg in LEAGUES)]
cols_partial = [col for col in all_null_cols if col not in cols_in_all]
ordered_cols = cols_in_all + cols_partial

print(f"Variables con nulos en al menos una liga: {len(all_null_cols)}")
print(f"  · Comunes a todas las ligas: {len(cols_in_all)}")
print(f"  · Específicas a alguna liga: {len(cols_partial)}")

#### 3.2.2 Visualización comparativa

In [ ]:
rows = []
for col in ordered_cols:
    row = {}
    for lg in LEAGUES:
        lg_cap = lg.capitalize()
        if col in null_data.get(lg, {}):
            row[f"{lg_cap}_Variable"] = col
            row[f"{lg_cap}_%"] = f"{null_data[lg][col]:.2f}%"
        else:
            row[f"{lg_cap}_Variable"] = ""
            row[f"{lg_cap}_%"] = ""
    rows.append(row)

null_comparison_df = pd.DataFrame(rows)

null_comparison_df.columns = pd.MultiIndex.from_tuples(
    [(lg.capitalize(), "Variable") if "_Variable" in col else (lg.capitalize(), "% nulo") 
     for lg in LEAGUES for col in [f"{lg.capitalize()}_Variable", f"{lg.capitalize()}_%"]]
)

display(
    null_comparison_df
    .style
    .hide(axis="index")
    .set_table_attributes('style="max-height:400px; overflow-y:auto; display:block;"')
)

### 3.3 Casas de apuestas estables en el core multi-league

Filtrado de variables de casas de apuestas del core multi-league según umbral máximo de valores nulos.

In [ ]:
# Prefijos de casas de apuestas según documentación oficial de football-data
bookmaker_prefixes = [
    "1XB", "B365", "BF", "BFD", "BMGM", "BV", "BS", "BW", 
    "CL", "GB", "IW", "LB", "PS", "SO", "SB", "SJ", 
    "SY", "VC", "WH"
]

odds_columns_core = [col for col in core_consolidated 
                     if any(col.startswith(book) for book in bookmaker_prefixes)]

THRESHOLD = 5.0
stable_bookmakers = []

for book in bookmaker_prefixes:
    book_cols = [c for c in odds_columns_core if c.startswith(book)]
    if not book_cols:
        continue
    
    max_null_pct = max(
        (dfs[lg][book_cols].isnull().sum().sum() / dfs[lg][book_cols].size) * 100
        for lg in LEAGUES
    )
    
    if max_null_pct <= THRESHOLD:
        stable_bookmakers.append({
            "Casa": book,
            "Columnas": len(book_cols),
            "Máx. % nulos": f"{max_null_pct:.2f}%",
            "Variables": ", ".join(sorted(book_cols))
        })

stable_df = pd.DataFrame(stable_bookmakers)

print(f"Umbral: ≤{THRESHOLD}% nulos | Casas de apuestas estables: {len(stable_df)}\n")
display(stable_df.style.hide(axis="index")) if len(stable_df) > 0 else print("Ninguna casa cumple el umbral")

### 3.4 Resumen de completitud

Consolidación de hallazgos sobre la calidad de datos del core multi-league.

In [ ]:
total_cols = len(core_consolidated)
cols_with_nulls = len(all_null_cols)
cols_clean = total_cols - cols_with_nulls
total_odds_cols = len(odds_columns_core)
stable_odds_cols = sum(stable_df["Columnas"])

print("\n── Resumen de completitud ─────────────────────────────────────────\n")
print(f"  {'Variables comunes':<22}{total_cols}")
print(f"  {'Sin nulos':<22}{cols_clean} ({(cols_clean/total_cols)*100:.2f}%)")
print(f"  {'Con nulos':<22}{cols_with_nulls} ({(cols_with_nulls/total_cols)*100:.2f}%)")

print("\n\n── Cobertura de cuotas ───────────────────────────────────────────\n")
print(f"  {'Columnas de cuotas':<22}{total_odds_cols}")
print(f"  {'Casas estables':<22}{len(stable_df)} (≤{THRESHOLD}% nulos)")
print(f"  {'Columnas utilizables':<22}{stable_odds_cols}/{total_odds_cols} ({(stable_odds_cols/total_odds_cols)*100:.2f}%)")
print("\n───────────────────────────────────────────────────────────────────")

---
##

## 4) Validaciones de integridad

Se realizan comprobaciones básicas de coherencia lógica y estructural sobre el core multi-league.


### 4.1 Consistencia estructural por temporada

Verificación del número esperado de partidos por temporada y del rango temporal cubierto en cada liga.

In [ ]:
for lg, df in dfs.items():
    dates = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
    min_date = dates.min().strftime("%d/%m/%Y")
    max_date = dates.max().strftime("%d/%m/%Y")
    total = len(df)
    
    season_counts = dates.dt.to_period("Y-JUL").astype(str).value_counts().sort_index()
    expected = season_counts.mode()[0]
    inconsistent = season_counts[season_counts != expected]
    
    if len(inconsistent) > 0:
        print(f"{lg.capitalize():<10}: {total:,} partidos | {min_date} → {max_date}")
        print(f"  ⚠ Inconsistencia: esperado {expected} partidos/temporada")
        for season, count in inconsistent.items():
            print(f"      {season}: {count} partidos")
    else:
        print(f"{lg.capitalize():<10}: {total:,} partidos ({expected}/temporada) | {min_date} → {max_date}")

### 4.2 Validación de resultados (`FTR`/`HTR`)

Se valida la consistencia entre resultados declarados y marcadores en el core multi-league.

In [ ]:
issues = sum(
    len(dfs[lg][
        ((dfs[lg]["FTR"] == "H") & (dfs[lg]["FTHG"] <= dfs[lg]["FTAG"])) |
        ((dfs[lg]["FTR"] == "A") & (dfs[lg]["FTAG"] <= dfs[lg]["FTHG"])) |
        ((dfs[lg]["FTR"] == "D") & (dfs[lg]["FTHG"] != dfs[lg]["FTAG"]))
    ])
    for lg in LEAGUES
)

print(f"Validación FTR/HTR: {issues} inconsistencias detectadas" if issues > 0 else "Validación FTR/HTR: todas las ligas correctas")

### 4.3 Detección de duplicados

Identificación de partidos duplicados por liga, utilizando como clave compuesta por `Date`, `HomeTeam` y `AwayTeam`.

In [ ]:
duplicates_found = False

for lg, df in dfs.items():
    dups = df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"], keep=False)
    
    if dups.sum() > 0:
        print(f"⚠ {lg.capitalize()}: {dups.sum()} filas duplicadas")
        display(df[dups][["Date", "HomeTeam", "AwayTeam", "FTR"]].head())
        duplicates_found = True

if not duplicates_found:
    print("Sin duplicados detectados en ninguna liga")

### 4.4 Control de valores negativos

Control de calidad para columnas numéricas del core multi-league donde no se esperan valores negativos.

In [ ]:
neg_found = False

for lg, df in dfs.items():
    numeric_cols = df[core_consolidated].select_dtypes(include="number").columns
    neg_counts = df[numeric_cols].lt(0).sum()
    neg_counts = neg_counts[neg_counts > 0].sort_values(ascending=False)
    
    if not neg_counts.empty:
        neg_found = True
        print(f"⚠ {lg.capitalize()}: valores negativos detectados")
        display(
            neg_counts.rename("Valores negativos")
                      .reset_index()
                      .rename(columns={"index": "Columna"})
                      .style.hide(axis="index")
        )

if not neg_found:
    print("Ningún valor negativo detectado en columnas numéricas del core consolidado")

### 4.5 Resumen de validaciones

Síntesis de los controles de integridad aplicados al core multi-league.

In [ ]:
total_partidos = sum(len(df) for df in dfs.values())

print("\n── Resumen de validaciones del core multi-league ───────────────────\n")
print(f"  {'Partidos totales validados':<30} {total_partidos:,}")
print(f"  {'Inconsistencias FTR/HTR':<30} {issues}")
print(f"  {'Duplicados detectados':<30} {sum(df.duplicated(subset=['Date','HomeTeam','AwayTeam'], keep=False).sum() for df in dfs.values())}")
print(f"  {'Valores negativos':<30} {'detectados' if neg_found else 'ninguno'}")
print("\n───────────────────────────────────────────────────────────────────")

---
##

## 5) Conclusiones del análisis exploratorio

El análisis comparativo cubre **10 temporadas** (2014/15–2023/24) de tres competiciones europeas, con un total de **10.660 partidos**: 3.800 de La Liga, 3.800 de la Premier League y 3.060 de la Bundesliga (18 equipos frente a 20 en las otras dos ligas).

Las tres competiciones presentan **rangos temporales alineados**, con temporadas que comienzan en agosto y finalizan entre mayo y junio del año siguiente.

### 5.1 Core multi-league

Se identificó un **core dataset de 43 variables comunes** a las tres ligas, con tipos de datos consistentes (sin drift entre competiciones). El core se estructura en:

| Grupo | Variables |
|-------|-----------|
| Identificación (4) | `Date`, `Div`, `HomeTeam`, `AwayTeam` |
| Resultados (6) | `FTHG`, `FTAG`, `FTR`, `HTHG`, `HTAG`, `HTR` |
| Estadísticas (12) | `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR` |
| Cuotas (21) | `B365`, `BW`, `IW`, `PS`, `VC`, `WH` (`H`/`D`/`A` + variantes) |

### 5.2 Completitud

- Las variables de **resultados y estadísticas** presentan **completitud total** en las tres ligas.
- Los valores nulos se concentran exclusivamente en **columnas de cuotas**, con un impacto global inferior al 1%.
- Las casas de apuestas con cobertura estable (≤5% nulos) en las tres ligas son las mismas identificadas en los EDA individuales: **B365, BW, PS, VC y WH**.

### 5.3 Integridad

- **Sin inconsistencias** en la validación FTR/HTR en ninguna liga.
- **Sin duplicados** detectados por clave compuesta (`Date`, `HomeTeam`, `AwayTeam`).
- **Sin valores negativos** en columnas numéricas del core.

### 5.4 Viabilidad del enfoque multi-liga para limpieza

El core consolidado de **43 variables** mantiene tipos consistentes, alta completitud y coherencia lógica en las tres competiciones. Los patrones de calidad detectados son idénticos entre ligas: nulos concentrados en cuotas, mismas casas estables (B365, BW, PS, VC, WH), validaciones de integridad superadas.

Esto permite aplicar un **pipeline de limpieza unificado** sobre el dataset multi-league, sin lógica específica por liga. La única diferencia estructural (306 vs 380 partidos/temporada) afecta al volumen, no al esquema.

En la siguiente fase se aplicarán las transformaciones y validaciones necesarias para generar el **dataset limpio consolidado** que servirá de base para la integración con el dataset de xG y el posterior modelado.

---
##

## 6) Exportación del core multi-league

### 6.1 Montaje del core multi-liga 

Concatenación de las tres ligas utilizando exclusivamente las columnas del core consolidado.

In [ ]:
for lg in LEAGUES:
    missing = set(core_consolidated) - set(dfs[lg].columns)
    if missing:
        raise ValueError(f"⚠ {lg}: columnas faltantes en core: {missing}")

df_core_consolidated = pd.concat(
    [dfs[lg][core_consolidated] for lg in LEAGUES],
    ignore_index=True
)

print(f"Core consolidado construido:")
print(f"  Filas: {len(df_core_consolidated):,} | Columnas: {len(core_consolidated)}")
print(f"  Ligas: {', '.join([lg.capitalize() for lg in LEAGUES])}")

### 6.2 Exportación a Parquet y metadatos

Guardado del core multi-league en formato Parquet con metadatos del esquema para fases posteriores.

In [ ]:
df_core_consolidated.to_parquet(CORE_MULTI_LEAGUE_VALIDATED_PATH, index=False)

metadata = {
    "num_columns": len(core_consolidated),
    "num_rows": len(df_core_consolidated),
    "num_leagues": len(LEAGUES),
    "leagues": LEAGUES,
    "columns": core_consolidated,
    "dtypes": {col: str(df_core_consolidated[col].dtype) for col in core_consolidated},
    "matches_per_league": {lg: len(dfs[lg]) for lg in LEAGUES}
}

with open(CORE_MULTI_LEAGUE_VALIDATED_SCHEMA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

core_rel = CORE_MULTI_LEAGUE_VALIDATED_PATH.relative_to(PROJECT_ROOT)
metadata_rel = CORE_MULTI_LEAGUE_VALIDATED_SCHEMA_PATH.relative_to(PROJECT_ROOT)

print(f"Archivos guardados:")
print(f"  · Dataset → {core_rel}")
print(f"  · Metadatos → {metadata_rel}")

---
##